In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
# Cargamos el dataset original para documentar el punto de partida
df_original = pd.read_csv("../src/data/listings.csv")

print(f"Registros originales: {len(df_original)}")

Registros originales: 9714


In [4]:
# Eliminamos la columna de neighbourhood_group, latitude y longitude
df_2025 = df_original.drop(columns=["neighbourhood_group", "latitude", "longitude"])

# Convertir 'last_review' a formato fecha
df_2025['last_review'] = pd.to_datetime(df_2025['last_review'], format='%Y-%m-%d')

# Filtramos para quedarnos con los registros relativos a 2025
df_2025 = df_2025[df_2025["last_review"].dt.year == 2025]

# Eliminamos los nulos
df_2025 = df_2025.dropna(subset=["price"])


In [5]:
df_2025.info()

<class 'pandas.DataFrame'>
Index: 6861 entries, 0 to 9688
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              6861 non-null   int64         
 1   name                            6861 non-null   str           
 2   host_id                         6861 non-null   int64         
 3   host_name                       6861 non-null   str           
 4   neighbourhood                   6861 non-null   str           
 5   room_type                       6861 non-null   str           
 6   price                           6861 non-null   float64       
 7   minimum_nights                  6861 non-null   int64         
 8   number_of_reviews               6861 non-null   int64         
 9   last_review                     6861 non-null   datetime64[us]
 10  reviews_per_month               6861 non-null   float64       
 11  calculated_host_list

In [6]:
df_2025.head(2)

,id,name,host_id,host_name,neighbourhood,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,96033,"Bonito piso a 200m de la playa, El Palo (Málaga)",510467,Rafael,Este,Entire home/apt,58.0,3,274,2025-09-29,1.88,1,324,40,ESFCTU0000290200003588210000000000000000VUT/MA...
1,166473,Perfect Location In Malaga,793360,Fred,Este,Private room,28.0,5,102,2025-03-27,0.59,5,288,3,NaN


In [7]:
# cargamos el nuevo dataset para añadir las columnas que nos interesan

df_detallado = pd.read_csv("../src/data/detallado.csv.gz")

In [8]:
# seleccionamos las columnas que queremos añadir al antiguo
df_extra = df_detallado[['id', 'accommodates', 'review_scores_rating']]

# Hacemos el merge con tu dataset antiguo
df_final = df_2025.merge(df_extra, on='id', how='left')

In [10]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 6861 entries, 0 to 6860
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              6861 non-null   int64         
 1   name                            6861 non-null   str           
 2   host_id                         6861 non-null   int64         
 3   host_name                       6861 non-null   str           
 4   neighbourhood                   6861 non-null   str           
 5   room_type                       6861 non-null   str           
 6   price                           6861 non-null   float64       
 7   minimum_nights                  6861 non-null   int64         
 8   number_of_reviews               6861 non-null   int64         
 9   last_review                     6861 non-null   datetime64[us]
 10  reviews_per_month               6861 non-null   float64       
 11  calculated_host

In [14]:
df = df_final.copy()

In [15]:
# precio medio por barrio
df.groupby("neighbourhood")["price"].mean().sort_values(ascending=False)

neighbourhood
Campanillas             2989.210526
Churriana                594.290909
Puerto de la Torre       444.931034
Este                     310.222061
Teatinos-Universidad     284.127660
Cruz De Humilladero      255.663265
Carretera de Cadiz       251.097598
Centro                   157.234670
Palma-Palmilla           141.650407
Ciudad Jardin            113.134615
Bailen-Miraflores         73.589862
Name: price, dtype: float64

In [16]:
# quitamos los outliers
umbral_extremo = df["price"].quantile(0.99)

df_outliers_extremos = df[df["price"] > umbral_extremo]

df_outliers_extremos["price"].describe()

count       69.000000
mean      8340.826087
std       3243.765708
min       1000.000000
25%       8500.000000
50%       9000.000000
75%       9858.000000
max      21911.000000
Name: price, dtype: float64

In [17]:
df["es_outlier"] = df["price"] > umbral_extremo

df_limpio = df[df["es_outlier"] == False]

In [18]:
df_limpio.groupby("neighbourhood")["price"].mean().sort_values(ascending=False)

neighbourhood
Campanillas             204.000000
Churriana               161.298077
Palma-Palmilla          141.650407
Puerto de la Torre      139.392857
Este                    137.897168
Centro                  117.459095
Ciudad Jardin           113.134615
Carretera de Cadiz      104.079632
Cruz De Humilladero      94.245675
Teatinos-Universidad     82.217391
Bailen-Miraflores        73.589862
Name: price, dtype: float64

In [20]:
df_limpio['precio_por_persona'] = df_limpio['price'] / df_limpio['accommodates']


In [ ]:
# esto cambia mucho el análisis

df_limpio.groupby("neighbourhood")["precio_por_persona"].mean().sort_values(ascending=False)


neighbourhood
Campanillas             37.555999
Palma-Palmilla          35.834563
Centro                  33.345216
Este                    32.363992
Churriana               29.367942
Carretera de Cadiz      29.009950
Cruz De Humilladero     28.722131
Ciudad Jardin           26.263193
Puerto de la Torre      25.083461
Bailen-Miraflores       24.165295
Teatinos-Universidad    20.659679
Name: precio_por_persona, dtype: float64

In [25]:
df_limpio[['precio_por_persona', 'review_scores_rating']].corr()


,precio_por_persona,review_scores_rating
precio_por_persona,1.000000,0.103266
review_scores_rating,0.103266,1.000000


In [36]:
(    df_limpio[df_limpio['neighbourhood'] == 'Centro']['accommodates']
    .value_counts(normalize=True) * 100
).round(2)


accommodates
4     35.14
2     28.48
3     11.55
6     10.23
5      7.27
8      2.50
1      1.89
7      1.41
10     0.41
9      0.28
12     0.28
16     0.13
14     0.13
13     0.11
11     0.11
15     0.07
Name: proportion, dtype: float64

In [37]:
(
    df_limpio[df_limpio['neighbourhood'] == 'Campanillas']['accommodates']
    .value_counts(normalize=True) * 100
).round(2)


accommodates
6     46.15
2     15.38
8      7.69
10     7.69
3      7.69
9      7.69
13     7.69
Name: proportion, dtype: float64

In [45]:
(    df_limpio[df_limpio['neighbourhood'] == 'Palma-Palmilla']['accommodates']
    .value_counts(normalize=True) * 100
).round(2)

accommodates
4     29.27
2     27.64
3     11.38
6     10.57
5      7.32
1      4.88
8      3.25
16     1.63
7      1.63
10     0.81
14     0.81
12     0.81
Name: proportion, dtype: float64

In [38]:
df_limpio.groupby("neighbourhood")["accommodates"].agg(lambda x: x.mode()[0])


neighbourhood
Bailen-Miraflores       4
Campanillas             6
Carretera de Cadiz      4
Centro                  4
Churriana               4
Ciudad Jardin           2
Cruz De Humilladero     2
Este                    4
Palma-Palmilla          4
Puerto de la Torre      4
Teatinos-Universidad    4
Name: accommodates, dtype: int64

In [39]:
df_limpio.groupby("neighbourhood")["accommodates"].agg(
    moda=lambda x: x.mode()[0],
    frecuencia=lambda x: (x == x.mode()[0]).sum()
)


,moda,frecuencia
neighbourhood,,
Bailen-Miraflores,4,76
Campanillas,6,6
Carretera de Cadiz,4,187
Centro,4,1615
Churriana,4,25
Ciudad Jardin,2,21
Cruz De Humilladero,2,84
Este,4,233
Palma-Palmilla,4,36


In [40]:
modas = df_limpio.groupby("neighbourhood")["accommodates"].agg(lambda x: x.mode()[0])


In [41]:
resultado = (
    df_limpio
    .groupby("neighbourhood")
    .apply(lambda g: g[g["accommodates"] == g["accommodates"].mode()[0]]["price"].median())
    .rename("mediana_precio_moda")
)


In [44]:
resultado = resultado.sort_values(ascending=False)
resultado

neighbourhood
Campanillas             250.5
Palma-Palmilla          132.5
Centro                  106.0
Este                    102.0
Churriana               100.0
Carretera de Cadiz       95.0
Puerto de la Torre       83.0
Bailen-Miraflores        81.5
Teatinos-Universidad     64.0
Cruz De Humilladero      62.5
Ciudad Jardin            53.0
Name: mediana_precio_moda, dtype: float64